In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.DeepPhys import DeepPhys
from neural_methods.model.TS_CAN import TSCAN

In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupA")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Headmotion/groupA")

# ----- video / signal params -----
VIDEO_FPS    = 30       # camera frame rate
PPG_FS       = 60       # PPG sensor sampling rate (Hz)

# ----- Group A preprocessing params -----
CHUNK_LENGTH = 180      # frames per clip (6 s at 30 fps)
IMG_H, IMG_W = 72, 72   # face crop resolution
LABEL_TYPE   = "DiffNormalized"  # needs cumsum in post-processing
DATA_FORMAT  = "NDCHW"  # (N, D, C, H, W)
NUM_CHANNELS = 6        # DiffNormalized (3) + Standardized (3)

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA
OUTPUT_DIR: /home/iec/MinhHieu/rPPG/results/Headmotion/groupA


In [3]:
# Model selection toggle
# Uncomment additional models as needed; each entry is (display_name, model_class_key, weight_path).
# model_class_key: "DeepPhys" or "Tscan"

MODELS = [
    ("PURE_DeepPhys",              "DeepPhys", "final_model_release/PURE_DeepPhys.pth"),
    ("PURE_TSCAN",                 "Tscan",    "final_model_release/PURE_TSCAN.pth"),
    ("SCAMPS_DeepPhys",          "DeepPhys", "final_model_release/SCAMPS_DeepPhys.pth"),
    ("SCAMPS_TSCAN",             "Tscan",    "final_model_release/SCAMPS_TSCAN.pth"),
    ("UBFC-rPPG_DeepPhys",       "DeepPhys", "final_model_release/UBFC-rPPG_DeepPhys.pth"),
    ("UBFC-rPPG_TSCAN",          "Tscan",    "final_model_release/UBFC-rPPG_TSCAN.pth"),
    ("BP4D_PseudoLabel_DeepPhys","DeepPhys", "final_model_release/BP4D_PseudoLabel_DeepPhys.pth"),
    ("BP4D_PseudoLabel_TSCAN",   "Tscan",    "final_model_release/BP4D_PseudoLabel_TSCAN.pth"),
    ("MA-UBFC_deepphys",         "DeepPhys", "final_model_release/MA-UBFC_deepphys.pth"),
    ("MA-UBFC_tscan",            "Tscan",    "final_model_release/MA-UBFC_tscan.pth"),
]

# TS-CAN frame_depth (TSM window size) — must match training config
TSCAN_FRAME_DEPTH = 10

print(f"Selected {len(MODELS)} model(s):")
for name, cls, path in MODELS:
    print(f"  {name}  ({cls})  ->  {path}")

Selected 10 model(s):
  PURE_DeepPhys  (DeepPhys)  ->  final_model_release/PURE_DeepPhys.pth
  PURE_TSCAN  (Tscan)  ->  final_model_release/PURE_TSCAN.pth
  SCAMPS_DeepPhys  (DeepPhys)  ->  final_model_release/SCAMPS_DeepPhys.pth
  SCAMPS_TSCAN  (Tscan)  ->  final_model_release/SCAMPS_TSCAN.pth
  UBFC-rPPG_DeepPhys  (DeepPhys)  ->  final_model_release/UBFC-rPPG_DeepPhys.pth
  UBFC-rPPG_TSCAN  (Tscan)  ->  final_model_release/UBFC-rPPG_TSCAN.pth
  BP4D_PseudoLabel_DeepPhys  (DeepPhys)  ->  final_model_release/BP4D_PseudoLabel_DeepPhys.pth
  BP4D_PseudoLabel_TSCAN  (Tscan)  ->  final_model_release/BP4D_PseudoLabel_TSCAN.pth
  MA-UBFC_deepphys  (DeepPhys)  ->  final_model_release/MA-UBFC_deepphys.pth
  MA-UBFC_tscan  (Tscan)  ->  final_model_release/MA-UBFC_tscan.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0 with Haar Cascade, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [8]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 10 subject folders

  S_001  video=S_001.mkv
  S_002  video=S_002.mkv
  S_003  video=S_003.mkv
  S_004  video=S_004.mkv
  S_005  video=S_005.mkv
  S_006  video=S_006.mkv
  S_007  video=S_007.mkv
  S_008  video=S_008.mkv
  S_009  video=S_009.mkv
  S_010  video=S_010.mkv

Total subjects: 10


In [9]:
# Data preprocessing
# Face crop -> DiffNormalized (3ch) + Standardized (3ch) = 6ch
# Chunk into clips of CHUNK_LENGTH frames, save as .npy

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Read and resample PPG
    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop + resize to 72x72
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # Compute DiffNormalized (3ch) and Standardized (3ch)
    diff_data = diff_normalize_data(frames_cropped)   # (T, H, W, 3)
    std_data  = standardized_data(frames_cropped)     # (T, H, W, 3)

    # Concatenate along channel axis -> 6 channels
    data_6ch = np.concatenate([diff_data, std_data], axis=-1)  # (T, H, W, 6)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)  # (T,)

    # Chunk into clips
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([data_6ch[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]    for i in range(clip_num)])

    # Save per-subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 6)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA

=== Processing S001 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=91
  15 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA/S001

=== Processing S002 ===
  Video: 2703 frames @ 30 fps
  PPG green: min=1, max=94
  15 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA/S002

=== Processing S003 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=2, max=95
  15 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA/S003

=== Processing S004 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=107
  15 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA/S004

=== Processing S005 ===
  Video: 2701 frames @ 30 fps
  PPG green: min=1, max=117
  15 clips -> /home/iec/MinhHieu/rPPG/preprocessed_data/Headmotion/groupA/S005

=== Processing S006 ===
  Video: 2702 frames @ 30 fps
  PPG green: min=1, max=105
  15 clips -> /home/iec/Min

In [10]:
# PyTorch Dataset  (NDCHW format, returns (D, 6, H, W) per clip)

class GroupADataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 6)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NDCHW: (D, H, W, 6) -> (D, 6, H, W)
        data = np.transpose(data, (0, 3, 1, 2))

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = GroupADataset(all_input_files)
loader  = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)

print(f"Dataset : {len(dataset)} clips")
print(f"Loader  : {len(loader)} batches (batch_size=4)")

Dataset : 150 clips
Loader  : 38 batches (batch_size=4)


In [11]:
# Post-processing functions

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Post-process predicted and label BVP chunk dicts into HR estimates.

    When diff_flag=True (DiffNormalized labels), signals are cumsum'd
    before detrending to recover the original BVP waveform.
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [12]:
# Main inference loop
# For each model in MODELS: load -> infer -> per-subject results -> aggregate metrics
# -> export metrics.json + ppg_results.csv into results_groupA/{model_name}/

FS = VIDEO_FPS

# lookup: subj_key -> subject info dict
subjects_by_key = {s["subj_key"]: s for s in subjects}

for model_display_name, model_class_key, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_display_name}  ({model_class_key})")
    print(f"Weights: {model_path}")
    print(f"{'='*70}\n")

    # ---- Instantiate model ----
    if model_class_key == "DeepPhys":
        model = DeepPhys(img_size=IMG_H)
        frame_depth = None  # no TSM alignment needed
    elif model_class_key == "Tscan":
        model = TSCAN(frame_depth=TSCAN_FRAME_DEPTH, img_size=IMG_H)
        frame_depth = TSCAN_FRAME_DEPTH
    else:
        raise ValueError(f"Unknown model_class_key: {model_class_key}")

    # ---- Load weights ----
    state_dict = torch.load(model_path, map_location=DEVICE)

    # strip 'module.' prefix if present (DataParallel artifact)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # ---- Inference ----
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Inference [{model_display_name}]"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch

            N, D, C, H, W = data_t.shape  # (N, CHUNK_LENGTH, 6, 72, 72)

            # Flatten batch + temporal -> (N*D, 6, H, W)
            data_flat   = data_t.view(N * D, C, H, W).to(DEVICE)
            labels_flat = labels_t.view(N * D)  # (N*D,)

            # For TS-CAN: trim to multiple of frame_depth (TSM alignment)
            if frame_depth is not None:
                trim = (N * D) // frame_depth * frame_depth
            else:
                trim = N * D  # DeepPhys: no alignment needed

            data_flat   = data_flat[:trim]
            labels_flat = labels_flat[:trim]

            pred = model(data_flat)            # (trim, 1)
            pred_np  = pred.squeeze(-1).cpu().numpy()      # (trim,)
            label_np = labels_flat[:trim].numpy()           # (trim,)

            for i in range(N):
                start = i * CHUNK_LENGTH
                end   = start + CHUNK_LENGTH
                if end > trim:
                    break

                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = pred_np[start:end]
                bvp_labels_dict[subj][cid] = label_np[start:end]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # ---- Per-subject results ----
    per_subject_results = []
    hr_preds_all  = []
    hr_labels_all = [] 
    snr_all       = []

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7}")
    print("-" * 55)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key],
            fs=FS, diff_flag=True
        )

        snr_db     = calculate_snr(pred_processed, hr_label, FS)
        hr_err     = hr_pred - hr_label

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "label_heartrate":     hr_label, 
            "heartrate_error":     hr_err,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        hr_labels_all.append(hr_label)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f}")

    hr_preds_all  = np.array(hr_preds_all)
    hr_labels_all = np.array(hr_labels_all)
    snr_all       = np.array(snr_all)

    # ---- Aggregate metrics ----
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - hr_labels_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))

    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))

    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\n--- Aggregate Metrics [{model_display_name}] ---")
    print(f"MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # ---- Export to per-model subdirectory ----
    model_output_dir = os.path.join(OUTPUT_DIR, model_display_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_display_name,
        "n_subjects": n,
        "evaluation_method": "FFT BVP-derived HR",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se,  "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":                r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "label_heartrate":     r["label_heartrate"], 
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # ---- Export ppg_results.csv ----
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"], 
            "heartrate_error":     r["heartrate_error"]
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "label_heartrate", "heartrate_error"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")


Model: PURE_DeepPhys  (DeepPhys)
Weights: /home/iec/MinhHieu/rPPG/final_model_release/PURE_DeepPhys.pth



/tmp/ipykernel_2941746/3690442588.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=DEVICE)


Model loaded. Total parameters: 2,228,643


Inference [PURE_DeepPhys]: 100%|██████████| 38/38 [01:20<00:00,  2.11s/it]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          36.475     85.693  -49.219   -8.94
S_002          62.402     86.572  -24.170  -11.55
S_003          49.219     76.025  -26.807   -4.18
S_004          56.689     97.559  -40.869   -8.94
S_005          45.264     87.451  -42.188   -9.61
S_006          61.523     61.963   -0.439   -2.73
S_007          37.354     94.922  -57.568  -10.88
S_008          58.887     56.689    2.197   -2.34
S_009          43.506     73.389  -29.883   -6.91
S_010          55.371     89.209  -33.838   -8.43

--- Aggregate Metrics [PURE_DeepPhys] ---
MAE     : 30.7178 +/- 5.5648 bpm
RMSE    : 35.4012 +/- 17.8135 bpm
MAPE    : 35.4631 +/- 6.0149 %
Pearson : -0.3611 +/- 0.3297
SNR     : -7.4508 +/- 0.9901 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupA/PURE_DeepP

...


Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          88.770     85.693    3.076   -4.28
S_002          49.219     86.572  -37.354  -11.68
S_003          78.662     76.025    2.637   -2.65
S_004          51.416     97.559  -46.143   -6.16
S_005          43.506     87.451  -43.945   -4.17
S_006          61.523     61.963   -0.439   -1.64
S_007          36.035     94.922  -58.887   -7.43
S_008          59.326     56.689    2.637    0.88
S_009          72.510     73.389   -0.879   -4.19


/tmp/ipykernel_2941746/3690442588.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=DEVICE)


S_010          56.689     89.209  -32.520   -9.13

--- Aggregate Metrics [MA-UBFC_deepphys] ---
MAE     : 22.8516 +/- 6.9160 bpm
RMSE    : 31.6309 +/- 19.1121 bpm
MAPE    : 25.2802 +/- 7.3885 %
Pearson : -0.3691 +/- 0.3286
SNR     : -5.0473 +/- 1.1055 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupA/MA-UBFC_deepphys/metrics.json
CSV saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupA/MA-UBFC_deepphys/ppg_results.csv

 name  predicted_heartrate  label_heartrate  heartrate_error
S_001            88.769531        85.693359         3.076172
S_002            49.218750        86.572266       -37.353516
S_003            78.662109        76.025391         2.636719
S_004            51.416016        97.558594       -46.142578
S_005            43.505859        87.451172       -43.945312
S_006            61.523438        61.962891        -0.439453
S_007            36.035156        94.921875       -58.886719
S_008            59.326172        56.689453         2.636719

Inference [MA-UBFC_tscan]: 100%|██████████| 38/38 [00:03<00:00,  9.92it/s]



Inference complete. Subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']

Subject       HR_pred   HR_label   HR_err     SNR
-------------------------------------------------------
S_001          86.133     85.693    0.439   -5.55
S_002          63.721     86.572  -22.852   -9.21
S_003          76.025     76.025    0.000   -2.69
S_004          62.402     97.559  -35.156   -6.27
S_005          87.451     87.451    0.000   -2.95
S_006          61.523     61.963   -0.439   -1.08
S_007          45.264     94.922  -49.658   -8.97
S_008          59.326     56.689    2.637   -2.45
S_009          50.537     73.389  -22.852   -5.84
S_010          39.990     89.209  -49.219   -7.96

--- Aggregate Metrics [MA-UBFC_tscan] ---
MAE     : 18.3252 +/- 6.1790 bpm
RMSE    : 26.7883 +/- 17.2589 bpm
MAPE    : 20.6930 +/- 6.6979 %
Pearson : -0.0141 +/- 0.3535
SNR     : -5.2959 +/- 0.8685 dB

Metrics saved to: /home/iec/MinhHieu/rPPG/results/Headmotion/groupA/MA-UBFC_ts

In [13]:
# convert metric.json to csv

import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc chứa các thư mục model dựa trên ảnh của bạn
ROOT_DIR = OUTPUT_DIR

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        model_name = data.get("model", "Unknown")
        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")

✅ Đã gom thành công 10 file JSON!
✅ File tổng hợp được lưu tại:
/home/iec/MinhHieu/rPPG/results/Headmotion/groupA/Model_Performance_Metrics.csv

Preview dữ liệu:
 rank            model  # subjects      MAE (bpm)      RMSE (bpm)       MAPE (%)        Pearson       SNR (dB)
    1  SCAMPS_DeepPhys          10 10.06 +/- 4.48 17.38 +/- 14.07 11.72 +/- 5.17  0.49 +/- 0.31 -5.60 +/- 0.75
    2  UBFC-rPPG_TSCAN          10 12.57 +/- 4.92 20.00 +/- 13.83 14.46 +/- 5.48  0.36 +/- 0.33 -4.49 +/- 1.01
    3       PURE_TSCAN          10 17.36 +/- 5.68 24.97 +/- 15.46 19.21 +/- 6.14 -0.13 +/- 0.35 -6.39 +/- 0.80
    4    MA-UBFC_tscan          10 18.33 +/- 6.18 26.79 +/- 17.26 20.69 +/- 6.70 -0.01 +/- 0.35 -5.30 +/- 0.87
    5 MA-UBFC_deepphys          10 22.85 +/- 6.92 31.63 +/- 19.11 25.28 +/- 7.39 -0.37 +/- 0.33 -5.05 +/- 1.11
